# NeuroVal-3D — Phase 2 on Kaggle

End-to-end demo of the validator stack on real radiology data, plus Stage 1 preprocessing on one BraTS 2020 volume.

**What this notebook does:**
1. Clones the project from your GitHub mirror
2. Downloads 369 TextBraTS reports (HuggingFace, MIT)
3. Downloads 1,007 RadGenome-Brain MRI reports (HuggingFace, research-only)
4. Runs the held-out perturbation benchmark on **TextBraTS**
5. Runs the held-out perturbation benchmark on **RadGenome-Brain MRI**
6. Runs Stage 1 preprocessing on one BraTS 2020 volume from the attached Kaggle dataset
7. Plots the headline AUROC bar chart vs BioClinicalBERT and RaTEScore-lite

Expected wall-clock with T4 GPU: ~20 minutes. CPU only: ~1.5 hours.


## Prerequisites

**Kaggle setup before running:**
1. Settings (right rail) → **Internet: ON**
2. Settings → **Accelerator: GPU T4 ×2** (or P100, either works)
3. **Add Data** (right rail) → search **`brats20-dataset-training-validation`** by user **awsaf49** → Add
4. Edit `GITHUB_URL` in the cell below to point to your fork/clone of the repo

**Direct dataset link:**
https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation


In [ ]:
# Project lives at vicky-16032005/neuroval3d (set 2026-04-30)
GITHUB_URL = "https://github.com/vicky-16032005/neuroval3d.git"

import os, subprocess, sys
PROJECT_DIR = "/kaggle/working/neuroval3d"

# 1. Heavy installs Kaggle doesn't ship by default
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai>=1.3", "nibabel>=5.2", "SimpleITK>=2.3", "python-dotenv"], check=True)

# 2. Clone the project
if not os.path.isdir(PROJECT_DIR):
    subprocess.run(["git", "clone", GITHUB_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)

# 3. Install our package + extras (data) for huggingface_hub etc.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[data,eval]"], check=True)

print("Project at:", PROJECT_DIR)
print(os.listdir(PROJECT_DIR))


In [ ]:
# Sanity check imports + GPU
import sys, torch, importlib
print(f"python: {sys.version.split()[0]}")
print(f"torch:  {torch.__version__}, cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}, vram: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Some Kaggle environments resolve `neuroval3d` to an empty namespace package despite
# the editable install succeeding. Force-prepend our src/ to sys.path and drop any
# stale module cache before the first real import.
_PROJECT_SRC = "/kaggle/working/neuroval3d/src"
if _PROJECT_SRC not in sys.path:
    sys.path.insert(0, _PROJECT_SRC)
for _m in list(sys.modules):
    if _m == "neuroval3d" or _m.startswith("neuroval3d."):
        del sys.modules[_m]

import neuroval3d
print(f"neuroval3d: v{getattr(neuroval3d, '__version__', '?')} from {neuroval3d.__file__}")


## 1 · Download real radiology reports from HuggingFace

TextBraTS (369) and RadGenome-Brain MRI (1,007) — both fully public, no auth needed.


In [ ]:
# This pulls all 369 TextBraTS reports into data/raw/TextBraTS/reports/
import subprocess, sys
subprocess.run([sys.executable, "scripts/download_textbrats_reports.py"], check=True)


In [ ]:
# This pulls 16 JSON files (5 disease subsets × 3 sections + split) into data/raw/RadGenome-BrainMRI/
import subprocess, sys
subprocess.run([sys.executable, "scripts/download_radgenome_reports.py"], check=True)


## 2 · Held-out benchmark on **TextBraTS** (369 reports)

70/30 split by `original_id` — fusion is trained on 258 base reports, evaluated on 111 unseen ones.


In [ ]:
from neuroval3d.evaluation import run_benchmark
from neuroval3d.data import textbrats_reports_only

tb_reports = textbrats_reports_only()
print(f"Loaded {len(tb_reports)} TextBraTS reports.")

result_tb = run_benchmark(
    reports=tb_reports,
    use_synthetic=False,
    n_samples=len(tb_reports),
    train_frac=0.7,
)
print(result_tb.summary_table())


## 3 · Held-out benchmark on **RadGenome-Brain MRI** (1,007 reports)

Larger, richer dataset — 5 disease subsets, explicit T1/T2/FLAIR mentions, all 7 perturbation ops fire.


In [ ]:
from neuroval3d.data import radgenome_reports_only

rg_reports = radgenome_reports_only(section="global_finding")
print(f"Loaded {len(rg_reports)} RadGenome reports.")

result_rg = run_benchmark(
    reports=rg_reports,
    use_synthetic=False,
    n_samples=len(rg_reports),
    train_frac=0.7,
)
print(result_rg.summary_table())


## 4 · Stage 1 preprocessing on a real BraTS 2020 volume

Loads four MRI modalities (T1, T1ce, T2, FLAIR) for one subject, runs N4 bias correction, brain masking, z-score normalisation, and reshapes to a 128³ cube. Saves a triptych preview of all four modalities.


In [ ]:
from pathlib import Path
from neuroval3d.data import Stage1Preprocessor, PreprocessingConfig

# The awsaf49 BraTS 2020 mirror normally mounts as
#   /kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/
# but if Kaggle ever re-nests it slightly we still want to find the right subjects dir.
def _find_brats_training_root() -> Path | None:
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    # Preferred: directory literally called MICCAI_BraTS2020_TrainingData
    for p in base.rglob("MICCAI_BraTS2020_TrainingData"):
        if p.is_dir():
            return p
    # Fallback: walk up from any BraTS20_Training_NNN subject dir
    for p in base.rglob("BraTS20_Training_001"):
        if p.is_dir():
            return p.parent
    return None

BRATS_ROOT = _find_brats_training_root()
if BRATS_ROOT is None:
    raise SystemExit(
        "BraTS dataset not found under /kaggle/input/. "
        "In Kaggle right-rail: Add Data \u2192 "
        "https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation"
    )
print(f"BraTS training root: {BRATS_ROOT}")
n_subjects = len(list(BRATS_ROOT.glob("BraTS20_Training_*")))
print(f"Subject directories found: {n_subjects}")

subject_id = "BraTS20_Training_001"
subject_dir = BRATS_ROOT / subject_id

# Files might be .nii or .nii.gz depending on the mirror; detect either.
def _modality_path(modality: str) -> str:
    for ext in (".nii", ".nii.gz"):
        cand = subject_dir / f"{subject_id}_{modality}{ext}"
        if cand.exists():
            return str(cand)
    raise FileNotFoundError(
        f"Missing {subject_id} {modality} under {subject_dir}; "
        f"contents: {sorted(p.name for p in subject_dir.iterdir())[:8] if subject_dir.exists() else 'dir missing'}"
    )

paths = {
    "T1":    _modality_path("t1"),
    "T1ce":  _modality_path("t1ce"),
    "T2":    _modality_path("t2"),
    "FLAIR": _modality_path("flair"),
}
seg_path = _modality_path("seg")
print("Resolved modality paths:")
for k, v in paths.items():
    print(f"  {k:5s}: {v}")

pre = Stage1Preprocessor(PreprocessingConfig(
    target_shape=(128, 128, 128),
    bias_correct=True,
    normalize="zscore",
))
out = pre.run(paths, seg_path=seg_path)

print(f"\nPreprocessed volume shape: {out.volume.shape}")
print(f"Brain-mask voxel count:    {int(out.brain_mask.sum()):,}")
print("Per-modality stats (pre-zscore):")
for m, s in out.modality_stats.items():
    print(f"  {m}: mean={s['mean']:.2f}, std={s['std']:.2f}")


In [ ]:
# Triptych: each modality, axial + coronal mid-slice
import matplotlib.pyplot as plt
import numpy as np

vol = out.volume
modalities = ["T1", "T1ce", "T2", "FLAIR"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, m in enumerate(modalities):
    D, H, W = vol.shape[1:]
    axes[0, i].imshow(vol[i, D // 2], cmap="gray")
    axes[0, i].set_title(f"{m} — axial")
    axes[0, i].axis("off")
    axes[1, i].imshow(vol[i, :, H // 2, :], cmap="gray", aspect="auto")
    axes[1, i].set_title(f"{m} — coronal")
    axes[1, i].axis("off")
fig.suptitle(f"Stage 1 preprocessing output — {subject}")
plt.tight_layout()
plt.savefig("/kaggle/working/preprocessing_triptych.png", dpi=120, bbox_inches="tight")
plt.show()


## 5 · Synthetic report generated from the segmentation mask

Demonstrates the templated VASARI feature extractor — same code that produces the synthetic warmup benchmark.


In [ ]:
from neuroval3d.data import SyntheticReportGenerator
import nibabel as nib, numpy as np

seg = np.asarray(nib.load(seg_path).dataobj, dtype=np.int16)
synth = SyntheticReportGenerator().from_mask(seg, voxel_volume_mm3=1.0)
print("=== Generated radiology report from the segmentation ===")
print(synth.text)
print("\n=== Extracted VASARI features ===")
for k, v in synth.vasari_features.items():
    print(f"  {k:25s}: {v}")


## 6 · The headline chart — NeuroVal-3D vs baselines


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

datasets = ["TextBraTS (n=369)", "RadGenome (n=1007)"]
ours = [
    result_tb.auroc_overall.get("fusion", 0.0),
    result_rg.auroc_overall.get("fusion", 0.0),
]
bioclin = [
    result_tb.auroc_overall.get("semantic", 0.0),
    result_rg.auroc_overall.get("semantic", 0.0),
]
ratescore = [
    result_tb.auroc_overall.get("ratescore_lite (baseline)", 0.0),
    result_rg.auroc_overall.get("ratescore_lite (baseline)", 0.0),
]

x = np.arange(len(datasets))
width = 0.27
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width, ours,      width, label="NeuroVal-3D fused (ours)", color="#2a9d8f")
ax.bar(x,         bioclin,   width, label="BioClinicalBERT (off-the-shelf)", color="#e9c46a")
ax.bar(x + width, ratescore, width, label="RaTEScore-lite (Jaccard baseline)", color="#e76f51")
for i, v in enumerate(ours):
    ax.text(x[i] - width, v + 0.02, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
for i, v in enumerate(bioclin):
    ax.text(x[i], v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
for i, v in enumerate(ratescore):
    ax.text(x[i] + width, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

ax.axhline(0.5, ls=":", color="gray", lw=1)
ax.text(-0.45, 0.51, "random", color="gray", fontsize=9)
ax.set_ylabel("Held-out AUROC")
ax.set_title("Brain-MRI hallucination detection — NeuroVal-3D vs off-the-shelf baselines")
ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.legend(loc="lower right")
ax.set_ylim([0, 1.10])
plt.tight_layout()
plt.savefig("/kaggle/working/auroc_bar_chart.png", dpi=120, bbox_inches="tight")
plt.show()


## 7 · Where the artefacts landed

Everything Kaggle keeps after this notebook commits ends up in `/kaggle/working/`. Two key files:

- `preprocessing_triptych.png` — Stage 1 output for one BraTS volume
- `auroc_bar_chart.png` — the headline chart

Plus all per-run AUROC tables under `/kaggle/working/neuroval3d/outputs/results/`.

## 8 · What's next

- **Cross-dataset transfer** (TextBraTS → RadGenome and reverse) is the next paper-table entry. Run `python scripts/run_cross_only.py` after this notebook completes; it shares the loaded BERT model.
- **Phase 2 proper**: train a small report decoder (BART-base) using the preprocessed volumes from Stage 1 above. Use the resulting generated reports as the test corpus for NeuroVal-3D — this closes the generator-validator loop.
- **Negation axis improvement** (round 6) — the fusion negation AUROC is the weakest spot at 0.63 on RadGenome. Integrating `negspaCy` should lift it past 0.85.
